In [1]:
from lxml import etree
from io import StringIO, BytesIO
from os.path import join, split, isdir
from os import getcwd
from glob import glob
import uuid
from datetime import datetime
import tifffile
import numpy as np

from ome_types.model import Instrument, Microscope, Objective, InstrumentRef, OME, Image, Pixels, Channel, TiffData, Detector
from ome_types import from_xml, to_xml

In [2]:
root_dir = r"."

image_name = split(getcwd())[-1]
export_dir = glob(join(root_dir, "*export*"))[0]
manufacturer = "Zeiss"
px_unit = "nm"

detector_1_name = "inLens"
detector_2_name = "SE2"

# The raw files are saved in directories corresponding to the FIBSEM detector used
assert isdir(join(root_dir, detector_1_name)), f"{detector_1_name} folder not found, are you sure this detector was used?"
assert isdir(join(root_dir, detector_2_name)), f"{detector_2_name} folder not found, are you sure this detector was used?"

In [3]:
# Finds the interpolation xml file in the export directory (the merge of the two detectors images interpolated at z positions)
tree = etree.parse(glob(join(root_dir, "*export*/*_interpolation.xml"))[0])
z_l = []
number_l = []
for vs in tree.findall("./VirtualSlice"):
    z_l.append(float(vs.find("./Z").text))
    number_l.append(int(vs.find("./Number").text))

# Get the Z pixel size
z_px = np.median(np.diff(z_l)) * 1000
size_z = len(z_l)


# Load XML in the first inLens image, extracting some metadata hiding in there.
root = None
with tifffile.TiffFile(glob(join(root_dir, f"{detector_1_name}/*.tif"))[0]) as tif:
    # Loop through all pages in the TIFF
    for page in tif.pages:
        for tag in page.tags.values():
            tag_name = tag.name
            tag_value = tag.value
            # Check if the tag contains XML
            if isinstance(tag_value, str) and tag_value.strip().startswith("<?xml"):
                # Parse the XML
                root = etree.fromstring(tag_value.encode('iso-8859-1'))

# To show all tags in the xml
# for elem in root.iter():
#     print(f"{elem.tag}: {elem.text}")

acq_date = datetime.fromisoformat(root.find(".//Date").text)
uscope_model = root.find(".//Machine").text.split("-")[0]
uscope_serial = root.find(".//Machine").text.split("-")[1]

# Extracting XY pixel size from two vectors each (even though Uy and Vx should be both equal to zero, that's still the theoretical way to do it)
Ux = float(root.find(".//Ux").text)
Uy = float(root.find(".//Uy").text)
Vx = float(root.find(".//Vx").text)
Vy = float(root.find(".//Vy").text)
x_px = np.sqrt(Ux**2 + Uy**2) * 1000
y_px = np.sqrt(Vx**2 + Vy**2) * 1000

with tifffile.TiffFile(glob(join(root_dir, "*export*/*.tif"))[0]) as tif:
    page = tif.pages[0]
    shape = page.shape
    dtype = page.dtype.__str__()
size_y, size_x = shape

detect_1 = Detector(model=detector_1_name)
detect_2 = Detector(model=detector_2_name)

print(f"µscope manufacturer: {manufacturer}")
print(f"µscope model: {uscope_model}")
print(f"µscope serial number: {uscope_serial}")
print(f"image name: {image_name}")
print(f"acquisition date: {acq_date}")
print()
print(f"pixel size: X:{x_px:.5f}   Y:{y_px:.5f}   Z:{z_px:.5f} {px_unit}")
print(f"image size: X:{size_x}   Y:{size_y}   Z:{size_z} px")
print(f"pixel type: {dtype}")

µscope manufacturer: Zeiss
µscope model: Crossbeam 550
µscope serial number: 8404010259
image name: Miri_FIBSEM
acquisition date: 2024-07-16 14:47:54.895000+02:00

pixel size: X:5.00008   Y:5.00008   Z:10.37654 nm
image size: X:4009   Y:3214   Z:838 px
pixel type: uint8


In [7]:
uscope = Microscope(manufacturer=manufacturer, model=uscope_model, serial_number=uscope_serial, type="Other")  # Read from metadata in a InLens image  (open image in notepad, search "crossbeam")
instrument = Instrument(id="Instrument:1", microscope=uscope, 
                        detectors=[detect_1, 
                                   detect_2])

tiff_data_blocks = []
for z, img_name in enumerate(glob(join(root_dir, export_dir, "*.tif"))):
    
    img_uuid = TiffData.UUID(file_name=f"{split(img_name)[1]}", value=f"urn:uuid:{uuid.uuid4()}")
    tiff_data_blocks.append(TiffData(uuid=img_uuid, ifd=0, first_z=z))

pixels = Pixels(id="Pixels:1", size_x=size_x, size_y=size_y, size_z=size_z, size_c=1, size_t=1, type=dtype, dimension_order="XYCZT",
                physical_size_x=x_px, physical_size_x_unit=px_unit, 
                physical_size_y=y_px, physical_size_y_unit=px_unit, 
                physical_size_z=z_px, physical_size_z_unit=px_unit, 
                tiff_data_blocks=tiff_data_blocks)


image = Image(acquisition_date=acq_date, pixels=pixels, id="Image:1", name=image_name, instrument_ref=InstrumentRef(id="Instrument:1"))

ome_meta = OME(uuid=f"urn:uuid:{uuid.uuid4()}")
ome_meta.images.append(image)
ome_meta.instruments.append(instrument)

ome_str = to_xml(ome_meta) #, validate=True
with open(join(root_dir, export_dir, image_name+".companion.ome"), "w", encoding="utf-8") as f:
    f.write(ome_str)